In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Reload environment variables from the .env file.
load_dotenv(override=True)

API_KEYS = {
    "openai": os.getenv("OPENAI_API_KEY"),
    "anthropic": os.getenv("ANTHROPIC_API_KEY"),
    "google": os.getenv("GOOGLE_API_KEY"),
    "deepseek": os.getenv("DEEPSEEK_API_KEY"),
    "groq": os.getenv("GROQ_API_KEY"),
    "grok": os.getenv("GROK_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY"),
}

ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROK_BASE_URL = "https://api.x.ai/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1/"

# Map provider names to their credentials and OpenAI-compatible endpoints.
PROVIDER_CONFIGS = {
    "openai": {
        "api_key": API_KEYS["openai"],
        "base_url": None,
    },
    "anthropic": {
        "api_key": API_KEYS["anthropic"],
        "base_url": ANTHROPIC_BASE_URL,
    },
    "deepseek": {
        "api_key": API_KEYS["deepseek"],
        "base_url": DEEPSEEK_BASE_URL,
    },
    "gemini": {
        "api_key": API_KEYS["google"],
        "base_url": GEMINI_BASE_URL,
    },
    "groq": {
        "api_key": API_KEYS["groq"],
        "base_url": GROQ_BASE_URL,
    },
    "grok": {
        "api_key": API_KEYS["grok"],
        "base_url": GROK_BASE_URL,
    },
    "openrouter": {
        "api_key": API_KEYS["openrouter"],
        "base_url": OPENROUTER_BASE_URL,
    },
}

llm_clients = {}

for provider, config in PROVIDER_CONFIGS.items():
    key = config["api_key"]
    url = config["base_url"]
    
    # Initialize client only if the corresponding API key is available
    if key:
        try:
            llm_clients[provider] = OpenAI(base_url=url, api_key=key)
            print(f" ▸ [{provider.upper()}] \033[92mEnabled Successfully\033[0m")
        except Exception as e:
            print(f" ▸ [{provider.upper()}] \033[91mInitialization Failed\033[0m: {e}")
    else:
        # Gracefully skip providers without API keys configured
        print(f" ▸ [{provider.upper()}] API Key not detected (Skipped)")

# Ollama requires a non-empty placeholder key for OpenAI SDK compatibility,
# but local Ollama does not authenticate or validate this value.
llm_clients["ollama"] = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama",
)

print(f"Configured providers: {', '.join(llm_clients)}")

In [ ]:
request = """
Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.
Not a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.
Include in your question that the answer must be short.
"""

request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

messages

In [ ]:
from IPython.display import Markdown, display

MODEL_NAME = "gpt-5.4-mini"

client = OpenAI()

# Generate a challenging reasoning question.
question_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
)

# Extract the generated question from the API response.
generated_question = question_response.choices[0].message.content.strip()

# Display the generated question as Markdown.
display(Markdown(generated_question))

In [ ]:
# Initialize state trackers for storing execution metrics
competitors = []
answers = []

# Prepare the conversation payload using the single generated question
messages = [{"role": "user", "content": generated_question}]

def record(model_name: str, answer: str) -> None:
    """
    Saves the execution output of a competitor model and renders it.
    
    Args:
        model_name (str): The identifier of the evaluated model.
        answer (str): The generated response text from the model.
    """
    # 1. State Accumulation: Store model metadata for downstream ranking
    competitors.append(model_name)
    answers.append(answer)
    
    # 2. Rich Text Render: Output formatted response in the notebook interface
    display(Markdown(answer))

In [ ]:
model_name = "gpt-5.4-nano"

response = llm_clients["openai"].chat.completions.create(model=model_name, messages=messages, reasoning_effort="none")
answer = response.choices[0].message.content

record(model_name, answer)

In [ ]:
model_name = "claude-sonnet-4-6"

response = llm_clients["anthropic"].chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

In [ ]:
model_name = "gemini-3.1-flash-lite"

response = llm_clients["gemini"].chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

In [ ]:
print(len(competitors))
print(competitors)
print(answers)

In [ ]:
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")
    print("-" * 100)

In [ ]:
together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
# judge = f"""You are judging a competition between {len(competitors)} competitors.
# Each model has been given this question:

# {generated_question}

# Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
# Respond with JSON, and only JSON, with the following format:
# {{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

# Here are the responses from each competitor:

# {together}

# Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [ ]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{generated_question}

Your job is to evaluate each response for clarity and strength of argument, give a score to each response(from 0 to 10), and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number, score", "second best competitor number, score", "third best competitor number, score ", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [ ]:
print(judge)

In [ ]:
judge_messages = [{"role": "user", "content": judge}]

model_name = "grok-4.3"

response = llm_clients["grok"].chat.completions.create(model=model_name, messages=judge_messages)
results = response.choices[0].message.content
print(results)

In [ ]:
import json

results_dict = json.loads(results)
ranks = results_dict["results"]
print(ranks)
for index, result in enumerate(ranks):
    competitor = competitors[int(result[0])-1]
    print(f"Rank {index+1}: {competitor}, score: {int(result[-1])}")

### Add Optimizer Part

In [ ]:
MAX_ITERATIONS = 3
MIN_ACCEPTABLE_SCORE = 7

MODEL_MAP = {
    "openai": "gpt-5.4-nano",
    "anthropic": "claude-sonnet-4-6",
    "gemini": "gemini-3.1-flash-lite"
}

print("=" * 60)
print(f" Initiating Evaluator-Optimizer Loop (Max Iterations: {MAX_ITERATIONS})")
print("=" * 60)

for iteration in range(MAX_ITERATIONS):
    print(f"\n" + "═" * 50)
    print(f"ROUND {iteration + 1} / {MAX_ITERATIONS}")
    print("═" * 50)
    
    # Step 1: Serialize Current Answers for the Judge
    together = ""
    for index, answer in enumerate(answers):
        together += f"# Response from competitor {index+1}\n\n"
        together += answer + "\n\n"
        
    # Step 2: Query Grok to Grade the Current Answers
    judge_prompt = f"""You are judging a competition between {len(competitors)} competitors.
    Each model has been given this question:

    {generated_question}

    Your job is to evaluate each response for clarity and strength of argument, give a score to each response(from 0 to 10), and rank them in order of best to worst.
    Respond with JSON, and only JSON, with the following format:
    {{"results": ["best competitor number, score", "second best competitor number, score", "third best competitor number, score ", ...]}}

    Here are the responses from each competitor:

    {together}

    Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""

    try:
        response = llm_clients["grok"].chat.completions.create(
            model="grok-4.3", 
            messages=[{"role": "user", "content": judge_prompt}]
        )
        results = response.choices[0].message.content.strip()
        results_dict = json.loads(results)
        ranks = results_dict["results"]
    except Exception as e:
        print(f" ❌ [Judge Error] Failed to evaluate in Round {iteration + 1}: {e}")
        break

    # Track if all competitors have successfully passed the score threshold in this round
    all_competitors_passed = True

    # Step 3: Check Scores and Optimize Underperforming Models
    for index, result in enumerate(ranks):
        competitor_id = int(result[0])
        score = int(result[-1])
        
        competitor_index = competitor_id - 1
        competitor_name = competitors[competitor_index]
        original_answer = answers[competitor_index]
        
        print(f"[{competitor_name.upper()}] scored: {score}/10")
        
        if score < MIN_ACCEPTABLE_SCORE:
            # We found at least one model that didn't pass yet
            all_competitors_passed = False
            print(f"Underperforming! Triggering optimizer for {competitor_name.upper()}...")
            
            optimization_prompt = f"""
            You previously provided this answer to the question '{generated_question}':
            "{original_answer}"
            
            An expert judge graded your answer as {score}/10 because it needs improvement.
            
            Please rewrite and optimize your answer. Make sure it is:
            1. Logically flawless and directly addresses the question.
            2. Extremely succinct (strictly under 30 words).
            
            Output only your final updated answer, nothing else. Do not include introductory text.
            """
            
            target_model = MODEL_MAP.get(competitor_name)
            
            if target_model:
                try:
                    opt_response = llm_clients[competitor_name].chat.completions.create(
                        model=target_model,
                        messages=[{"role": "user", "content": optimization_prompt}]
                    )
                    optimized_answer = opt_response.choices[0].message.content.strip()
                    
                    # Update global state in-place for the next iteration round
                    answers[competitor_index] = optimized_answer
                    print(f"[Updated] {competitor_name.upper()} registered new optimized draft.")
                except Exception as e:
                    print(f"[Optimization Failed] for {competitor_name}: {e}")
        else:
            print(f"[Passed] {competitor_name.upper()} is accepted.")

    # Step 4: Early Exit Guard
    if all_competitors_passed:
            print("\n" + "═" * 50)
            print("SUCCESS: All competitors have met the MIN_ACCEPTABLE_SCORE! Stopping loop early.")
            print("═" * 50)
            break
else:
    print("\n" + "═" * 50)
    print("REACHED LIMIT: Maximum iterations completed. Loop finished.")
    print("═" * 50)